# RQ2, Part 3: Real Analysis (Era-Moderation + Reassignment Effect)

Combines the real outputs of Parts 1 and 2. Runs the real moderated OLS regression (priority x era, comments x era) and the real reassignment-effect analysis, both on the **full population** (N=27,385, from `num_reassignments_FULL.csv`), superseding the earlier N=3,000 subsample version.

**Run this only after Parts 1 and 2 have both completed** and their output CSVs are present in this session.

In [1]:
!pip install -q pandas numpy requests statsmodels scipy || pip install -q pandas numpy requests statsmodels scipy --break-system-packages

In [5]:
"""
RQ2 Completion: Real num_reassignments analysis, merged with real JIRA data.
Real, executed analysis on real data uploaded from Colab notebook 24.
"""
import pandas as pd
import statsmodels.formula.api as smf

jira = pd.read_csv("apache_jira_raw.csv")
reassign = pd.read_csv("num_reassignments_FULL.csv")  # UPDATED: full population, not the N=3,000 subsample
merged = jira.merge(reassign, on="issue_id", how="inner")
print(f"Real merged sample (FULL population): {len(merged)} issues with both resolution_time and num_reassignments")

merged["resolution_date_parsed"] = pd.to_datetime(merged["resolution_date"], errors="coerce", utc=True)
merged["created_parsed"] = pd.to_datetime(merged["created"], errors="coerce", utc=True)
merged["era"] = (merged["resolution_date_parsed"] >= "2023-01-01").map({True: "ai_era", False: "pre_ai"})
merged["resolution_time_days"] = (merged["resolution_date_parsed"] - merged["created_parsed"]).dt.total_seconds() / 86400
merged["era_binary"] = (merged["era"] == "ai_era").astype(int)
merged = merged.dropna(subset=["resolution_time_days", "priority", "num_comments", "num_reassignments"])
merged = merged[merged["resolution_time_days"] >= 0]
print(f"Real usable sample after cleaning: {len(merged)}")
print(merged["era"].value_counts())

orig_model = smf.ols("resolution_time_days ~ (C(priority) + num_comments) * era_binary", data=merged).fit()
ext_model = smf.ols("resolution_time_days ~ (C(priority) + num_comments + num_reassignments) * era_binary", data=merged).fit()

print("\nFull extended model, all era-interaction terms:")
for term in ext_model.pvalues.index:
    if "era_binary" in term and term != "era_binary":
        print(f"  {term}: coef={ext_model.params[term]:.3f}, p={ext_model.pvalues[term]:.4f}")

print(f"\nnum_reassignments main effect: coef={ext_model.params['num_reassignments']:.2f} days/reassignment, p={ext_model.pvalues['num_reassignments']:.6f}")
print(f"Original model R2: {orig_model.rsquared:.4f}, Extended model R2: {ext_model.rsquared:.4f}")
f2 = (ext_model.rsquared - orig_model.rsquared) / (1 - ext_model.rsquared)
print(f"Incremental effect size (f2): {f2:.4f}")


Real merged sample (FULL population): 27385 issues with both resolution_time and num_reassignments
Real usable sample after cleaning: 27385
era
pre_ai    22571
ai_era     4814
Name: count, dtype: int64

Full extended model, all era-interaction terms:
  C(priority)[T.Critical]:era_binary: coef=-18.683, p=0.8263
  C(priority)[T.Major]:era_binary: coef=2.366, p=0.9236
  C(priority)[T.Minor]:era_binary: coef=24.654, p=0.3210
  C(priority)[T.Trivial]:era_binary: coef=-24.824, p=0.6407
  num_comments:era_binary: coef=15.074, p=0.0000
  num_reassignments:era_binary: coef=-16.778, p=0.0116

num_reassignments main effect: coef=45.16 days/reassignment, p=0.000000
Original model R2: 0.0243, Extended model R2: 0.0366
Incremental effect size (f2): 0.0128
